In [ ]:
import pandas as pd
import numpy as np
import time
import tensorflow as tf

from tensorflow import keras
from keras.models import Model
from keras.layers import Input, Dense
from keras.callbacks import EarlyStopping

from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
caminho_pasta_tratado = '../../dataset tratado/lycos-cicids2017/'

nome_dados_treinamento = 'Redução de Dimensionalidade/lycos_cicids2017_treinamento_reduzidos.csv'
nome_dados_teste       = 'Redução de Dimensionalidade/lycos_cicids2017_teste_reduzidos.csv'

In [ ]:
print("Carregando dataset de treinamento...")
df_treino_full = pd.read_csv(caminho_pasta_tratado + nome_dados_treinamento)
print(f"Dataset completo: {df_treino_full.shape}")

df_treino_benign = df_treino_full[df_treino_full['Label'] == 'benign'].copy()
print(f"Apenas BENIGN: {df_treino_benign.shape}")

X_treino = df_treino_benign.drop('Label', axis=1).values

display(df_treino_benign.head())

In [ ]:
# Arquitetura do Autoencoder
# Encoder: n → 32 → 16 → 8 | Decoder: 8 → 16 → 32 → n
n_features = X_treino.shape[1]

inputs = Input(shape=(n_features,))

# Encoder
encoded = Dense(32, activation='relu')(inputs)
encoded = Dense(16, activation='relu')(encoded)
encoded = Dense(8, activation='relu')(encoded)

# Decoder
decoded = Dense(16, activation='relu')(encoded)
decoded = Dense(32, activation='relu')(decoded)
outputs = Dense(n_features, activation='sigmoid')(decoded)

autoencoder = Model(inputs, outputs)
autoencoder.compile(optimizer='adam', loss='mse')
autoencoder.summary()

In [ ]:
# Treinamento do Autoencoder (apenas em tráfego BENIGN)
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

print("Iniciando treinamento do Autoencoder...")
inicio_treino = time.time()

history = autoencoder.fit(
    X_treino, X_treino,
    epochs=50,
    batch_size=256,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=1
)

fim_treino = time.time()
print(f"\nTreinamento concluído! Tempo total: {fim_treino - inicio_treino:.2f} segundos.")

In [ ]:
# Definindo o limiar de anomalia (percentil 95 do erro de reconstrução no treino BENIGN)
X_treino_rec = autoencoder.predict(X_treino, verbose=0)
erros_treino = np.mean(np.power(X_treino - X_treino_rec, 2), axis=1)

PERCENTIL = 95
limiar = np.percentile(erros_treino, PERCENTIL)

print(f"Estatísticas do erro de reconstrução (BENIGN treino):")
print(f"  Média:   {erros_treino.mean():.6f}")
print(f"  Desvio:  {erros_treino.std():.6f}")
print(f"  Máximo:  {erros_treino.max():.6f}")
print(f"  Limiar ({PERCENTIL}º percentil): {limiar:.6f}")

In [ ]:
def avaliar_autoencoder(autoencoder, limiar, df_teste, nome_cenario, benign_label):
    X_teste = df_teste.drop('Label', axis=1).values
    y_teste_real = df_teste['Label'].values

    X_teste_rec = autoencoder.predict(X_teste, verbose=0)
    erros_teste = np.mean(np.power(X_teste - X_teste_rec, 2), axis=1)

    y_pred_binario = np.where(erros_teste > limiar, 'ATTACK', 'BENIGN')
    y_real_binario = np.where(y_teste_real == benign_label, 'BENIGN', 'ATTACK')

    labels_bin = ['BENIGN', 'ATTACK']
    cm = confusion_matrix(y_real_binario, y_pred_binario, labels=labels_bin)
    cm_df = pd.DataFrame(
        cm,
        index=[f"Real_{l}" for l in labels_bin],
        columns=[f"Pred_{l}" for l in labels_bin]
    )

    print(f"\n{'='*60}")
    print(f"CENÁRIO: {nome_cenario}")
    print(f"Total de amostras: {len(y_teste_real)}")
    print(f"  BENIGN: {(y_real_binario == 'BENIGN').sum()}")
    print(f"  ATTACK: {(y_real_binario == 'ATTACK').sum()}")
    print(f"\n=== MATRIZ DE CONFUSÃO ===")
    display(cm_df.style.format("{:.0f}"))
    print(f"\n=== RELATÓRIO DE MÉTRICAS ===")
    print(classification_report(y_real_binario, y_pred_binario, labels=labels_bin, zero_division=0))

    return y_real_binario, y_pred_binario, cm

In [ ]:
CLASS_ALIASES_LATEX = {'BENIGN': 'BENIGN', 'ATTACK': 'ATTACK'}


def escape_latex(value):
    replacements = {
        "\\": "\\textbackslash{}",
        "&": "\\&",
        "%": "\\%",
        "$": "\\$",
        "#": "\\#",
        "_": "\\_",
        "{": "\\{",
        "}": "\\}",
        "~": "\\textasciitilde{}",
        "^": "\\textasciicircum{}",
    }
    return "".join(replacements.get(char, char) for char in str(value))


def format_confusion_value(value, is_diagonal):
    value = int(value)
    if is_diagonal:
        return f"\\ok{{{value}}}"
    if value != 0:
        return f"\\err{{{value}}}"
    return "0"


def make_latex_confusion_matrix(cm_values, class_labels, caption, table_label):
    headers = [escape_latex(CLASS_ALIASES_LATEX.get(l, l)) for l in class_labels]
    rows = []
    for i, real_label in enumerate(class_labels):
        row_values = [format_confusion_value(cm_values[i][j], i == j) for j in range(len(class_labels))]
        rows.append((f"Real\\_{escape_latex(CLASS_ALIASES_LATEX.get(real_label, real_label))}", row_values))

    first_col_width = max([0] + [len(row_name) for row_name, _ in rows])
    col_widths = [max(len(headers[i]), *(len(values[i]) for _, values in rows)) for i in range(len(headers))]

    def format_row(first_cell, values):
        first = first_cell.ljust(first_col_width)
        rest = " & ".join(str(value).ljust(col_widths[i]) for i, value in enumerate(values))
        return f"            {first} & {rest} \\\\"

    lines = [
        "\\begin{table}[H]",
        "    \\centering",
        "    \\small",
        f"        \\begin{{tabular}}{{l|{'r' * len(class_labels)}}}",
        "            \\hline",
        format_row("", headers),
        "            \\hline",
    ]
    lines.extend(format_row(row_name, row_values) for row_name, row_values in rows)
    lines.extend([
        "            \\hline",
        "        \\end{tabular}",
        "    }",
        f"    \\caption{{{escape_latex(caption)}}}",
        f"    \\label{{{table_label}}}",
        "\\end{table}",
    ])
    return "\n".join(lines)


def format_metric(value):
    return "-" if value is None else f"{value:.2f}"


def make_latex_metrics_report(y_true_values, y_pred_values, class_labels, caption, table_label):
    report = classification_report(
        y_true_values, y_pred_values,
        labels=class_labels, output_dict=True, zero_division=0,
    )
    total_support = int(sum(report[label]["support"] for label in class_labels))
    rows = []
    for label in class_labels:
        metrics = report[label]
        rows.append([
            escape_latex(label),
            format_metric(metrics["precision"]),
            format_metric(metrics["recall"]),
            format_metric(metrics["f1-score"]),
            str(int(metrics["support"])),
        ])

    rows.extend([
        ["\\textbf{Acurácia}", "-", format_metric(report["accuracy"]), "-", str(total_support)],
        ["\\textbf{Média Macro}", format_metric(report["macro avg"]["precision"]), format_metric(report["macro avg"]["recall"]), format_metric(report["macro avg"]["f1-score"]), str(total_support)],
        ["\\textbf{Média Ponderada}", format_metric(report["weighted avg"]["precision"]), format_metric(report["weighted avg"]["recall"]), format_metric(report["weighted avg"]["f1-score"]), str(total_support)],
    ])

    headers = ["Classe", "Precisão", "Revocação", "F1-score", "Suporte"]
    col_widths = [max(len(str(row[i])) for row in [headers] + rows) for i in range(len(headers))]

    def format_row(values):
        return "        " + " & ".join(str(value).ljust(col_widths[i]) for i, value in enumerate(values)) + " \\\\"

    lines = [
        "\\begin{table}[H]",
        "    \\centering",
        "    \\small",
        "    \\begin{tabular}{lrrrr}",
        "        \\hline",
        format_row(headers),
        "        \\hline",
    ]
    lines.extend(format_row(row) for row in rows[:len(class_labels)])
    lines.extend([
        "        \\hline",
        format_row(rows[-3]),
        format_row(rows[-2]),
        format_row(rows[-1]),
        "        \\hline",
        "    \\end{tabular}",
        f"    \\caption{{{escape_latex(caption)}}}",
        f"    \\label{{{table_label}}}",
        "\\end{table}",
    ])
    return "\n".join(lines)

In [ ]:
df_teste = pd.read_csv(caminho_pasta_tratado + nome_dados_teste)
y_real, y_pred, cm = avaliar_autoencoder(autoencoder, limiar, df_teste, 'Teste Completo', 'benign')

labels_bin = ['BENIGN', 'ATTACK']

print(make_latex_confusion_matrix(
    cm, labels_bin,
    'Autoencoder — LycoS-IDS2017 com MDI (Teste Completo) — Matriz de Confusão',
    'table:ae_lycos_mdi_completo_mc',
))
print()
print(make_latex_metrics_report(
    y_real, y_pred, labels_bin,
    'Autoencoder — LycoS-IDS2017 com MDI (Teste Completo) — Relatório de Métricas',
    'table:ae_lycos_mdi_completo_metricas',
))